# Replicação Fiel do Artigo: VGG-16 nos 6 Cenários da Tabela 3
Este notebook executa o treinamento e a avaliação da CNN **VGG-16** treinada do zero (`weights=None`), exatamente como na metodologia do artigo científico:
> *"Preprocessing and Optimizer Impact on Image-Based Autoimmune Disease Diagnosis"* (RITA, 2025).

Ele cobre os **6 cenários da Tabela 3 do artigo** cruzando os fatores:
1. **Técnicas de Imagem:** Original (`none`), Expansão de Contraste (`contrast_expansion`) e Equalização de Histograma (`histogram_equalization`).
2. **Otimizadores:** `SGD` e `Adam`.

### 1. Verificar GPU e Atualizar Repositório
Certifique-se de que a GPU está ativa (**Ambiente de Execução > Alterar tipo de ambiente de execução > T4 GPU**).

In [7]:
!nvidia-smi

Thu Nov 19 12:22:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   48C    P8             18W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [6]:
import os
if not os.path.exists("/content/Projeto-PDI"):
    !git clone https://github.com/orleoncio/Projeto-PDI.git /content/Projeto-PDI

%cd /content/Projeto-PDI
!git pull origin main

/content/Projeto-PDI
From https://github.com/orleoncio/Projeto-PDI
 * branch            main       -> FETCH_HEAD
Already up to date.


### 2. Instalar Dependências

In [8]:
!pip install -q scikit-learn pandas pillow torchvision torch

### 3. Validar a Arquitetura da VGG-16

In [ ]:
%cd /content/Projeto-PDI
!python -u tests/test_vgg16.py

## 🧪 CENÁRIO 1: Imagem Original (Baseline)
Comparação entre SGD e Adam sem filtros na imagem de entrada.

#### 1.1 VGG-16 Original + SGD (Meta: 98.16% | Obtido anteriormente: ~89.67%)

In [ ]:
%cd /content/Projeto-PDI
!python -u main_vgg16.py --optimizer sgd --preprocessing none --epochs 30 --batch-size 32 --amp --num-workers 2

#### 1.2 VGG-16 Original + Adam (Meta: 19.96% | Obtido anteriormente: 21.73% - Colapso Confirmado)

In [ ]:
%cd /content/Projeto-PDI
!python -u main_vgg16.py --optimizer adam --preprocessing none --epochs 30 --batch-size 32 --amp --num-workers 2

## 🧪 CENÁRIO 2: Expansão de Contraste (Contrast Stretching)
O pré-processamento que, segundo o artigo, **resgata o Adam do colapso** e eleva a acurácia para mais de 86%.

#### 2.1 VGG-16 Contraste + SGD (Meta do Artigo: 97.76% acc | 97.58% F1)

In [ ]:
%cd /content/Projeto-PDI
!python -u main_vgg16.py --optimizer sgd --preprocessing contrast_expansion --epochs 30 --batch-size 32 --amp --num-workers 2

#### 2.2 VGG-16 Contraste + Adam (Meta do Artigo: 86.32% acc | 86.35% F1 — Fenômeno de Resgate!)

In [9]:
%cd /content/Projeto-PDI
!python -u main_vgg16.py --optimizer adam --preprocessing contrast_expansion --epochs 30 --batch-size 32 --amp --num-workers 2

/content/Projeto-PDI
Dispositivo: cuda
Arquitetura: VGG-16 (ImageNet fine-tuning) | Pré-processamento: contrast_expansion | Otimizador: ADAM | Épocas: 30 | Batch size: 32 | Augment: False | AMP: True
Imagens de treinamento: 10877
Imagens de teste:       2720
Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth
100% 528M/528M [00:02<00:00, 187MB/s]
Epoch [1/30] Loss: 1.0801 Accuracy: 58.17%
Epoch [2/30] Loss: 0.7070 Accuracy: 73.65%
Epoch [3/30] Loss: 0.5771 Accuracy: 78.67%
Epoch [4/30] Loss: 0.5286 Accuracy: 81.44%
Epoch [5/30] Loss: 0.4990 Accuracy: 82.70%
Epoch [6/30] Loss: 0.4896 Accuracy: 82.60%
Epoch [7/30] Loss: 0.4695 Accuracy: 83.37%
Epoch [8/30] Loss: 0.4432 Accuracy: 85.07%
Epoch [9/30] Loss: 0.4024 Accuracy: 86.08%
Epoch [10/30] Loss: 0.3865 Accuracy: 86.86%
Epoch [11/30] Loss: 0.3757 Accuracy: 87.06%
Epoch [12/30] Loss: 0.3474 Accuracy: 88.15%
Epoch [13/30] Loss: 0.3270 Accuracy: 88.75%
Epoch [14/30]

## 🧪 CENÁRIO 3: Equalização de Histograma (Histogram Equalization)
Redistribuição uniforme de intensidade de brilho em cada canal da imagem.

#### 3.1 VGG-16 Equalização + SGD (Meta do Artigo: 96.80% acc | 96.73% F1)

In [10]:
%cd /content/Projeto-PDI
!python -u main_vgg16.py --optimizer sgd --preprocessing histogram_equalization --epochs 30 --batch-size 32 --amp --num-workers 2

/content/Projeto-PDI
Dispositivo: cuda
Arquitetura: VGG-16 (ImageNet fine-tuning) | Pré-processamento: histogram_equalization | Otimizador: SGD | Épocas: 30 | Batch size: 32 | Augment: True | AMP: True
Imagens de treinamento: 10877
Imagens de teste:       2720
Epoch [1/30] Loss: 0.6930 Train Acc: 73.63% | Test Acc: 85.15% ★ [BEST]
Epoch [2/30] Loss: 0.3546 Train Acc: 87.08% | Test Acc: 89.52% ★ [BEST]
Epoch [3/30] Loss: 0.2746 Train Acc: 90.29% | Test Acc: 91.18% ★ [BEST]
Epoch [4/30] Loss: 0.2332 Train Acc: 91.43% | Test Acc: 92.79% ★ [BEST]
Epoch [5/30] Loss: 0.2066 Train Acc: 92.97% | Test Acc: 92.90% ★ [BEST]
Epoch [6/30] Loss: 0.1923 Train Acc: 93.18% | Test Acc: 92.90%
Epoch [7/30] Loss: 0.1706 Train Acc: 94.04% | Test Acc: 92.98% ★ [BEST]
Epoch [8/30] Loss: 0.1657 Train Acc: 94.39% | Test Acc: 95.11% ★ [BEST]
Epoch [9/30] Loss: 0.1430 Train Acc: 94.99% | Test Acc: 93.97%
Epoch [10/30] Loss: 0.1450 Train Acc: 95.15% | Test Acc: 94.63%
Epoch [11/30] Loss: 0.1343 Train Acc: 95.61% 

#### 3.2 VGG-16 Equalização + Adam (Meta do Artigo: 20.81% acc | 5.74% F1 — Colapso Mantido)

In [11]:
%cd /content/Projeto-PDI
!python -u main_vgg16.py --optimizer adam --preprocessing histogram_equalization --epochs 30 --batch-size 32 --amp --num-workers 2

/content/Projeto-PDI
Dispositivo: cuda
Arquitetura: VGG-16 (do zero) | Pré-processamento: histogram_equalization | Otimizador: ADAM | Épocas: 30 | Batch size: 32 | Augment: False | AMP: True
Imagens de treinamento: 10877
Imagens de teste:       2720
Epoch [1/30] Loss: 8.7422 Accuracy: 19.45%
Epoch [2/30] Loss: 1.7290 Accuracy: 20.33%
Epoch [3/30] Loss: 1.7301 Accuracy: 20.42%
Epoch [4/30] Loss: 1.7297 Accuracy: 20.08%
Epoch [5/30] Loss: 1.7292 Accuracy: 19.59%
Epoch [6/30] Loss: 1.7296 Accuracy: 20.12%
Epoch [7/30] Loss: 1.7285 Accuracy: 20.68%
Epoch [8/30] Loss: 1.7287 Accuracy: 20.48%
Epoch [9/30] Loss: 1.7294 Accuracy: 20.64%
Epoch [10/30] Loss: 1.7286 Accuracy: 20.78%
Epoch [11/30] Loss: 1.7288 Accuracy: 20.07%
Epoch [12/30] Loss: 1.7288 Accuracy: 20.75%
Epoch [13/30] Loss: 1.7289 Accuracy: 20.23%
Epoch [14/30] Loss: 1.7285 Accuracy: 20.61%
Epoch [15/30] Loss: 1.7281 Accuracy: 21.04%
Epoch [16/30] Loss: 1.7290 Accuracy: 20.04%
Epoch [17/30] Loss: 1.7290 Accuracy: 20.41%
Epoch [18/3

## 📊 Confrontação Completa com a Tabela 3 do Artigo
Carrega os 6 modelos treinados e gera a tabela completa comparando as métricas obtidas contra os valores oficiais publicados.

In [12]:
import torch
import pandas as pd
from data.hep2_dataset import create_dataloaders
from models.vgg16 import create_vgg16
from training.test import test_model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Valores da Tabela 3 do artigo para a VGG-16 nos 6 cenários
paper_data = {
    ("Original", "SGD"): {"acc": 98.16, "f1": 98.05, "prec": 98.14, "rec": 97.98},
    ("Original", "Adam"): {"acc": 19.96, "f1": 5.55, "prec": 3.33, "rec": 16.67},
    ("Contraste", "SGD"): {"acc": 97.76, "f1": 97.58, "prec": 97.60, "rec": 97.56},
    ("Contraste", "Adam"): {"acc": 86.32, "f1": 86.35, "prec": 86.69, "rec": 86.69},
    ("Equalização", "SGD"): {"acc": 96.80, "f1": 96.73, "prec": 96.76, "rec": 96.71},
    ("Equalização", "Adam"): {"acc": 20.81, "f1": 5.74, "prec": 3.47, "rec": 16.67},
}

configs = [
    ("Original", "SGD", "vgg16_original_sgd.pth", "none"),
    ("Original", "Adam", "vgg16_original_adam.pth", "none"),
    ("Contraste", "SGD", "vgg16_contrast_expansion_sgd.pth", "contrast_expansion"),
    ("Contraste", "Adam", "vgg16_contrast_expansion_adam.pth", "contrast_expansion"),
    ("Equalização", "SGD", "vgg16_histogram_equalization_sgd.pth", "histogram_equalization"),
    ("Equalização", "Adam", "vgg16_histogram_equalization_adam.pth", "histogram_equalization"),
]

rows = []
for label_prep, label_opt, pth_file, prep_arg in configs:
    path = f"results/{pth_file}"
    try:
        use_norm = (label_opt == "SGD" or prep_arg == "contrast_expansion")
        _, test_loader = create_dataloaders(batch_size=32, preprocessing=prep_arg, normalize=use_norm)
        model = create_vgg16(num_classes=6, pretrained=False).to(device)
        model.load_state_dict(torch.load(path, map_location=device, weights_only=True))

        print(f"\n>>> Testando VGG-16 | Pré-processamento: {label_prep} | Otimizador: {label_opt} <<<")
        res = test_model(model, test_loader, device)

        p = paper_data[(label_prep, label_opt)]
        rows.append({
            "Pré-processamento": label_prep,
            "Otimizador": label_opt,
            "Acurácia Treino (%)": f"{res['accuracy']*100:.2f}%",
            "Acurácia Artigo (%)": f"{p['acc']:.2f}%",
            "Dif Acurácia (%)": f"{(res['accuracy']*100 - p['acc']):+.2f}%",
            "F1 Treino (%)": f"{res['f1']*100:.2f}%",
            "F1 Artigo (%)": f"{p['f1']:.2f}%",
            "Dif F1 (%)": f"{(res['f1']*100 - p['f1']):+.2f}%"
        })
    except FileNotFoundError:
        print(f"[-] Pesos {path} ainda não treinados/encontrados.")

if rows:
    df = pd.DataFrame(rows)
    print("\n===================== TABELA COMPARATIVA VGG-16 (TODOS OS CENÁRIOS) =====================")
    display(df)


[-] Pesos results/vgg16_original_sgd.pth ainda não treinados/encontrados.
[-] Pesos results/vgg16_original_adam.pth ainda não treinados/encontrados.
[-] Pesos results/vgg16_contrast_expansion_sgd.pth ainda não treinados/encontrados.

>>> Testando VGG-16 | Pré-processamento: Contraste | Otimizador: Adam <<<

===== RESULTADOS DO TESTE =====
Accuracy:  87.06%
Precision: 88.43%
Recall:    85.60%
F1-score:  86.76%

>>> Testando VGG-16 | Pré-processamento: Equalização | Otimizador: SGD <<<

===== RESULTADOS DO TESTE =====
Accuracy:  97.39%
Precision: 97.28%
Recall:    96.97%
F1-score:  97.12%

>>> Testando VGG-16 | Pré-processamento: Equalização | Otimizador: Adam <<<

===== RESULTADOS DO TESTE =====
Accuracy:  21.73%
Precision: 3.62%
Recall:    16.67%
F1-score:  5.95%

===================== TABELA COMPARATIVA VGG-16 (TODOS OS CENÁRIOS) =====================


,Pré-processamento,Otimizador,Acurácia Treino (%),Acurácia Artigo (%),Dif Acurácia (%),F1 Treino (%),F1 Artigo (%),Dif F1 (%)
0,Contraste,Adam,87.06%,86.32%,+0.74%,86.76%,86.35%,+0.41%
1,Equalização,SGD,97.39%,96.80%,+0.59%,97.12%,96.73%,+0.39%
2,Equalização,Adam,21.73%,20.81%,+0.92%,5.95%,5.74%,+0.21%
